# Streaming Kafka : INGTopic

Un **producer** envoie un nombre aléatoire (0-100) chaque seconde dans le topic `INGTopic`.
Un **consumer** lit ces nombres et met à jour un graphique en temps réel.

Ordre : exécuter les cellules 1, 2 puis 3. Pour arrêter : bouton ■ (Interrupt), puis cellule 4.

## Cellule 1 : configuration

In [ ]:
import json, random, threading, time
from collections import deque

from kafka import KafkaProducer, KafkaConsumer
import matplotlib.pyplot as plt
from IPython.display import clear_output

# Les 3 brokers : si l'un est en panne, le client contacte les autres
BOOTSTRAP = ['master:9092', 'datanode1:9092', 'datanode2:9092']
TOPIC = 'INGTopic'

## Cellule 2 : producer (en arrière-plan)

In [ ]:
producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

stop_event = threading.Event()

def produce():
    while not stop_event.is_set():
        n = random.randint(0, 100)
        producer.send(TOPIC, {'value': n, 'ts': time.time()})
        time.sleep(1)

# Thread : la boucle infinie tourne sans bloquer le notebook
producer_thread = threading.Thread(target=produce, daemon=True)
producer_thread.start()
print('Producer démarré : un nombre par seconde vers', TOPIC)

## Cellule 3 : consumer + graphique en temps réel

In [ ]:
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=BOOTSTRAP,
    group_id='ing-plot',
    auto_offset_reset='latest',
    value_deserializer=lambda b: json.loads(b.decode('utf-8'))
)

values = deque(maxlen=50)   # les 50 derniers nombres

try:
    for msg in consumer:
        values.append(msg.value['value'])
        clear_output(wait=True)
        plt.figure(figsize=(10, 4))
        plt.plot(list(values), marker='o')
        plt.ylim(0, 100)
        plt.title(f"INGTopic : dernier nombre = {values[-1]}  "
                  f"(partition {msg.partition}, offset {msg.offset})")
        plt.xlabel('Messages reçus')
        plt.ylabel('Valeur')
        plt.grid(True)
        plt.show()
except KeyboardInterrupt:
    print('Consumer arrêté')

## Cellule 4 : arrêt (après avoir interrompu la cellule 3)

In [ ]:
stop_event.set()
producer.flush()
producer.close()
consumer.close()
print('Producer et Consumer arrêtés')